In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel

tokenizer = AutoTokenizer.from_pretrained('/home/lick/project/pro_TCMLLM/BaseModels/Qwen/Qwen3-1.7B', use_fast=False, trust_remote_code=True)
model = AutoModelForCausalLM.from_pretrained('/home/lick/project/pro_TCMLLM/BaseModels/Qwen/Qwen3-1.7B', device_map='auto', torch_dtype=torch.bfloat16)
model = PeftModel.from_pretrained(model, '/home/lick/project/pro_TCMLLM/output/Qwen3-1.7B/checkpoint-1084')
model.eval()

# 测试完整的对话格式
text = '<|im_start|>system\n你是一个医学专家，你需要根据用户的问题，给出带有思考的回答。<|im_end|>\n<|im_start|>user\n医生，我最近胃部不适<|im_end|>\n<|im_start|>assistant\n<think>'
print(f'Input text: {repr(text)}')

input_ids = tokenizer.encode(text, return_tensors='pt').to('cuda')

with torch.no_grad():
    output = model.generate(
        input_ids=input_ids,
        max_new_tokens=50,
        do_sample=False,
        repetition_penalty=1.0,
        pad_token_id=tokenizer.eos_token_id,
    )

new_tokens = output[0][len(input_ids[0]):]
response = tokenizer.decode(new_tokens, skip_special_tokens=False)
print(f'Generated tokens: {new_tokens[:10].tolist()}')
print(f'Response: {repr(response[:100])}')

/home/lick/tools/anaconda3/envs/TCMLLM/lib/python3.10/site-packages/bitsandbytes/cuda_setup/main.py:136: UserWarning: /home/lick/tools/anaconda3 did not contain libcudart.so as expected! Searching further paths...
  warn(msg)
/home/lick/tools/anaconda3/envs/TCMLLM/lib/python3.10/site-packages/bitsandbytes/cuda_setup/main.py:136: UserWarning: WARNING: The following directories listed in your path were found to be non-existent: {PosixPath('/home/lick/.cache/dotnet_bundle_extract')}
  warn(msg)
/home/lick/tools/anaconda3/envs/TCMLLM/lib/python3.10/site-packages/bitsandbytes/cuda_setup/main.py:136: UserWarning: WARNING: The following directories listed in your path were found to be non-existent: {PosixPath('"en","osLocale"'), PosixPath('"en","defaultMessagesFile"'), PosixPath('{"userLocale"'), PosixPath('"/home/lick/.cursor-server/bin/c25eb90df95d64f6d280779237c1ca39f9f3eef0/out/nls.messages.json","locale"'), PosixPath('"en","resolvedLanguage"'), PosixPath('"en","availableLanguages"'), Pos


===================================BUG REPORT===================================
Welcome to bitsandbytes. For bug reports, please submit your error trace to: https://github.com/TimDettmers/bitsandbytes/issues
CUDA_SETUP: WARNING! libcudart.so not found in any environmental path. Searching /usr/local/cuda/lib64...
CUDA SETUP: CUDA runtime path found: /usr/local/cuda/lib64/libcudart.so
CUDA SETUP: Highest compute capability among GPUs detected: 8.9
CUDA SETUP: Detected CUDA version 120
CUDA SETUP: Loading binary /home/lick/tools/anaconda3/envs/TCMLLM/lib/python3.10/site-packages/bitsandbytes/libbitsandbytes_cuda120.so...


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

We've detected an older driver with an RTX 4000 series GPU. These drivers have issues with P2P. This can affect the multi-gpu inference when using accelerate device_map.Please make sure to update your driver to the latest version which resolves this.
The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.


Input text: '<|im_start|>system\n你是一个医学专家，你需要根据用户的问题，给出带有思考的回答。<|im_end|>\n<|im_start|>user\n医生，我最近胃部不适<|im_end|>\n<|im_start|>assistant\n<think>'
Generated tokens: [106287, 3837, 20002, 36587, 42411, 104044, 100518, 32948, 108684, 3837]
Response: '嗯，用户说他最近胃部不适，想了解应该怎么做。首先，我需要回忆一下胃部不适的常见原因和处理方法。胃部不适可能有很多原因，比如消化不良、胃炎、胃溃疡，或者压力'


: 